# M3L2 E12 - RAG desde cero con LangChain

## Objetivo

En este notebook vas a construir un RAG completo desde cero y paso a paso.

RAG significa **Retrieval Augmented Generation**. En castellano: generacion aumentada con recuperacion.

La idea es simple:

1. Tenemos informacion propia en documentos.
2. Buscamos los fragmentos relevantes para una pregunta.
3. Le pasamos esos fragmentos al modelo.
4. El modelo responde usando ese contexto.

## Entregable

Al terminar deberias tener una chain RAG que:

- crea documentos de ejemplo,
- los divide en chunks,
- calcula embeddings,
- guarda los vectores en FAISS,
- recupera fragmentos relevantes,
- arma un prompt con contexto,
- llama al modelo,
- devuelve una respuesta final como string,
- permite inspeccionar cada parte del flujo.

## Mapa mental del RAG

```text
                 INGESTION

Texto crudo -> Document -> Splitter -> Chunks -> Embeddings -> Vector Store
                                                             |
                                                             v
                                                        Retriever

                  CONSULTA

Pregunta -> Retriever -> Docs relevantes -> Contexto -> Prompt -> LLM -> Parser -> Respuesta
```

Un error comun es pensar que RAG es "hacer una pregunta al modelo".

RAG no empieza en el modelo. RAG empieza antes: en como preparamos, partimos, indexamos y recuperamos documentos.

## Glosario minimo

| Termino | Que significa | En LangChain |
|---|---|---|
| Corpus | Conjunto total de textos disponibles | lista de documentos |
| Document | Texto + metadata | `Document(page_content=..., metadata=...)` |
| Metadata | Datos extra del documento | fuente, categoria, fecha |
| Chunk | Fragmento pequeno de un documento | salida del splitter |
| Splitter | Componente que divide texto | `RecursiveCharacterTextSplitter` |
| Embedding | Vector numerico que representa significado | `OpenAIEmbeddings` |
| Vector store | Base donde se guardan vectores | `FAISS` |
| Retriever | Interfaz para buscar docs relevantes | `as_retriever()` |
| k | Cantidad de docs a recuperar | `search_kwargs={"k": 3}` |
| Contexto | Texto recuperado que recibe el LLM | string con chunks |
| Prompt | Instruccion final al modelo | `ChatPromptTemplate` |
| Parser | Convierte salida del modelo | `StrOutputParser` |

## Por que RAG ayuda

Un LLM puede responder con conocimiento general, pero no conoce necesariamente tus documentos privados.

RAG reduce ese problema porque obliga al sistema a traer contexto antes de responder.

| Sin RAG | Con RAG |
|---|---|
| El modelo responde con memoria interna | El modelo recibe documentos relevantes |
| Puede inventar mas facil | Tiene contexto concreto |
| No se puede auditar de donde salio la respuesta | Podemos ver que chunks se recuperaron |
| Todo depende del prompt | El pipeline separa busqueda y generacion |

In [ ]:
# En Colab, ejecuta esta celda si faltan paquetes.
# !pip install langchain langchain-openai langchain-community faiss-cpu

In [ ]:
import os
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")

print("API key cargada para esta sesion.")

## Paso 1 - Imports

Cada import cumple una funcion concreta dentro del pipeline:

| Import | Para que sirve |
|---|---|
| `Document` | Crear documentos con texto y metadata |
| `RecursiveCharacterTextSplitter` | Partir documentos en chunks |
| `OpenAIEmbeddings` | Convertir texto en vectores |
| `FAISS` | Guardar y buscar vectores |
| `ChatPromptTemplate` | Armar el prompt RAG |
| `ChatOpenAI` | Llamar al modelo |
| `StrOutputParser` | Obtener string final |
| `RunnablePassthrough` | Pasar la pregunta original dentro de LCEL |

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

## Paso 2 - Crear el corpus

Para que el ejercicio sea facil de subir a Colab, no vamos a depender de archivos externos.

Creamos documentos en memoria con `Document`.

La `metadata` importa porque despues nos permite saber de que fuente vino cada fragmento.

In [ ]:
# TODO 1: crear una lista llamada documentos con 4 objetos Document.
# Tema sugerido: politicas internas de una empresa.
# Cada Document debe tener page_content y metadata con al menos {"fuente": "..."}.

documentos = []

print("Documentos:", len(documentos))
for doc in documentos:
    print(doc.metadata, "->", doc.page_content[:80])

## Paso 3 - Dividir en chunks

Un documento largo no se suele mandar entero al modelo.

Primero lo dividimos en fragmentos.

Parametros clave:

| Parametro | Que controla |
|---|---|
| `chunk_size` | Tamano aproximado de cada fragmento |
| `chunk_overlap` | Solapamiento entre fragmentos para no cortar ideas |

Si los chunks son enormes, recuperamos mucho ruido.
Si son demasiado chicos, perdemos contexto.

In [ ]:
# TODO 2: crear RecursiveCharacterTextSplitter.
# Usa chunk_size=180 y chunk_overlap=30.

splitter = None
chunks = None

print("Chunks generados:", len(chunks))
for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print("Metadata:", chunk.metadata)
    print(chunk.page_content)

## Paso 4 - Crear embeddings

Un embedding es una lista de numeros que representa significado.

LangChain usa el modelo de embeddings para convertir cada chunk en vector.

Luego FAISS guarda esos vectores para buscar por similitud semantica.

In [ ]:
# TODO 3: crear embeddings con OpenAIEmbeddings().

embeddings = None

# Debug: convertir una frase en vector y mirar su longitud.
vector_demo = None
print("Dimension del vector:", len(vector_demo))
print("Primeros 5 valores:", vector_demo[:5])

## Paso 5 - Crear el vector store

El vector store guarda chunks + embeddings.

En este ejercicio usamos FAISS porque es simple y funciona en memoria.

```text
chunks -> embeddings -> FAISS index
```

In [ ]:
# TODO 4: crear vectorstore con FAISS.from_documents(chunks, embeddings).

vectorstore = None
print(type(vectorstore))

## Paso 6 - Convertir FAISS en retriever

El retriever es la interfaz estandar de busqueda.

En vez de llamar a metodos especificos de FAISS, usamos:

```python
retriever.invoke(pregunta)
```

Eso vuelve el sistema mas intercambiable.

In [ ]:
# TODO 5: crear retriever con k=3.

retriever = None

pregunta = "Cuantos dias de vacaciones tengo?"
docs_recuperados = None

for i, doc in enumerate(docs_recuperados, start=1):
    print(f"\nDoc recuperado {i}")
    print("Fuente:", doc.metadata.get("fuente"))
    print(doc.page_content)

## Paso 7 - Formatear contexto

El modelo no recibe objetos `Document`.

El modelo recibe texto.

Por eso necesitamos convertir la lista de documentos recuperados en un string de contexto.

In [ ]:
# TODO 6: completar format_docs para unir page_content y fuente.

def format_docs(docs):
    pass


contexto = format_docs(docs_recuperados)
print(contexto)

## Paso 8 - Crear el prompt RAG

El prompt debe dejar clara la regla central:

> Responde usando solamente el contexto. Si no alcanza, dilo.

Esto no elimina todos los errores, pero baja mucho la probabilidad de inventar.

In [ ]:
# TODO 7: crear ChatPromptTemplate con variables {contexto} y {pregunta}.

prompt = None
print(prompt)

## Paso 9 - Componer la chain RAG con LCEL

La chain completa tiene dos entradas internas:

```text
pregunta -> retriever -> format_docs -> contexto
pregunta -----------------------------> pregunta
```

Luego ambas entran al prompt.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# TODO 8: completar rag_chain usando LCEL.
# Pista:
# {
#   "contexto": retriever | format_docs,
#   "pregunta": RunnablePassthrough(),
# } | prompt | llm | parser

rag_chain = None
respuesta = None
print(respuesta)

## Paso 10 - Debugging del RAG

Cuando un RAG falla, no mires solo la respuesta final.

Revisa estas preguntas:

| Pregunta de debugging | Que inspeccionar |
|---|---|
| Se cargaron documentos? | `len(documentos)` |
| Se partieron bien? | imprimir chunks |
| El retriever trae algo relevante? | `retriever.invoke(...)` |
| El contexto tiene la respuesta? | `format_docs(...)` |
| El prompt limita al modelo? | texto del prompt |

In [ ]:
# TODO 9: prueba otra pregunta dentro del corpus y otra fuera del corpus.

pregunta_en_corpus = None
pregunta_fuera_del_corpus = None

print("Dentro del corpus:")
print(rag_chain.invoke(pregunta_en_corpus))

print("\nFuera del corpus:")
print(rag_chain.invoke(pregunta_fuera_del_corpus))

## Checklist final

- [ ] Cree documentos con metadata.
- [ ] Dividi los documentos en chunks.
- [ ] Cree embeddings.
- [ ] Cree un vector store FAISS.
- [ ] Cree un retriever.
- [ ] Inspeccione documentos recuperados.
- [ ] Formatee contexto.
- [ ] Cree prompt RAG.
- [ ] Compuse la chain con LCEL.
- [ ] Probe una pregunta respondible y una no respondible.

In [ ]:
assert len(documentos) >= 4
assert len(chunks) > 0
assert embeddings is not None
assert vectorstore is not None
assert retriever is not None
assert isinstance(contexto, str) and len(contexto) > 0
assert rag_chain is not None
assert isinstance(respuesta, str)
print("Checks OK")

## Resumen

RAG no es una sola funcion. Es un pipeline:

```text
preparar documentos -> indexar -> recuperar -> construir contexto -> generar respuesta
```

LangChain ayuda porque cada parte queda como componente separado, testeable y reemplazable.